In [7]:
# Import library require for process
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
import warnings as ws
ws.filterwarnings('ignore')
from sklearn.decomposition import KernelPCA




# kernel_pca_selection - input -> indep_X and output dep_Y and n_components
# Kernel PCA is unsupervised (doesn't use dep_Y), but kept as a parameter for consistency
# with the rest of the pipeline's function signatures.
# Feature scaling MUST happen before Kernel PCA, since it's variance-based like PCA.
# Unlike PCA, Kernel PCA uses a kernel trick to capture non-linear relationships,
# so it does NOT expose explained_variance_ratio_ the same way regular PCA does.
def kernel_pca_selection(indep_X, dep_Y, n_components, kernel='rbf'):
    sc = StandardScaler()
    X_scaled = sc.fit_transform(indep_X)
    kpca = KernelPCA(n_components=n_components, kernel=kernel)
    X_kpca = kpca.fit_transform(X_scaled)
    print("Eigenvalues:", kpca.eigenvalues_)
    print("Kernel used:", kernel)
    kpca_df = pd.DataFrame(
        X_kpca,
        columns=[f'KPC{i+1}' for i in range(n_components)],
        index=indep_X.index
    )
    return kpca_df
   
    
#split_scalar - Split the input, output train and test set. then changes the input to scalar value    
def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)    
        return X_train, X_test, y_train, y_test

# r2_prediction - used for regression method, model prediction evaluate method
def r2_prediction(regressor,X_test,y_test):
     y_pred = regressor.predict(X_test)
     from sklearn.metrics import r2_score
     r2=r2_score(y_test,y_pred)
     return r2
    
# Linear method is used for Linear regression model creation and r2 prediction
def Linear(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.linear_model import LinearRegression
        regressor = LinearRegression()
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2   

# SVM_Linear method is used for svm model creation and r2 prediction
def svm_linear(X_train,y_train,X_test):
                
        from sklearn.svm import SVR
        regressor = SVR(kernel = 'linear')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
# svm_NL method is used for svm_NL model creation and r2 prediction   
def svm_NL(X_train,y_train,X_test):
                
        from sklearn.svm import SVR
        regressor = SVR(kernel = 'rbf')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     
# Decision method is used for Decision tree model creation and r2 prediction   
def Decision(X_train,y_train,X_test):
        
        # Fitting K-NN to the Training setC
        from sklearn.tree import DecisionTreeRegressor
        regressor = DecisionTreeRegressor(random_state = 0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     
# random method is used for random forest model creation and r2 prediction  
def random(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.ensemble import RandomForestRegressor
        regressor = RandomForestRegressor(n_estimators = 10, random_state = 0)
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2 
    
# selectk_regression method is used for create dataset with columns name as 'Linear','SVMl','SVMnl','Decision','Random' and index ChiSquare
# and fill the each columns values 
def pca_regression(acclin,accsvml,accsvmnl,accdes,accrf): 
    
    dataframe=pd.DataFrame(index=['Kernel PCA'],columns=['Linear','SVMl','SVMnl','Decision','Random'
                                                                                     ])

    for number,idex in enumerate(dataframe.index):
        
        dataframe['Linear'][idex]=acclin[number]       
        dataframe['SVMl'][idex]=accsvml[number]
        dataframe['SVMnl'][idex]=accsvmnl[number]
        dataframe['Decision'][idex]=accdes[number]
        dataframe['Random'][idex]=accrf[number]
    return dataframe
    

In [8]:
# Read data from file and datatset should without index
dataset=pd.read_csv("prep.csv",index_col=None)
df2=dataset
# Preprocessed by one hot encoding
df2 = pd.get_dummies(df2, drop_first=True)
# assign the input only
indep_X=df2.drop('classification_yes', axis=1)
# assign output only
dep_Y=df2['classification_yes']

# choose the feature selection here using 5 feature 
kbest=kernel_pca_selection(indep_X,dep_Y,7)      


Eigenvalues: [48.76238991 16.39774153 10.65825596  8.9786758   8.0653548   6.71579116]
Kernel used: rbf


In [9]:
# Create 5 empty list for each algorithm and split the input and output
# Evalute each algorithmwise r2 score and send selectk_regression funtion
# finally the evalution data represent by table view.
acclin=[]
accsvml=[]
accsvmnl=[]
accdes=[]
accrf=[]

X_train, X_test, y_train, y_test=split_scalar(kbest,dep_Y)  
for i in kbest:  
   
    r2_lin=Linear(X_train,y_train,X_test)
    acclin.append(r2_lin)
    
    r2_sl=svm_linear(X_train,y_train,X_test)    
    accsvml.append(r2_sl)
    
    r2_NL=svm_NL(X_train,y_train,X_test)
    accsvmnl.append(r2_NL)
    
    r2_d=Decision(X_train,y_train,X_test)
    accdes.append(r2_d)
    
    r2_r=random(X_train,y_train,X_test)
    accrf.append(r2_r)
    
    
result=pca_regression(acclin,accsvml,accsvmnl,accdes,accrf)

In [10]:
result
# 6

,Linear,SVMl,SVMnl,Decision,Random
Kernel PCA,0.894529,0.892024,0.958609,0.956597,0.978733
